# CST wake-potential fit: TD-VF versus resonators

This notebook fits the same truncated CST wake-potential data with Time-Domain Vector Fitting and the original IDEFIX resonator model. Both reconstructions are evaluated over the complete available time interval.

TD-VF treats the normalized Gaussian bunch profile as input and the CST wake potential as output. Therefore the RMS bunch duration `SIGMA_T` and the bunch center must match the CST simulation. If CST specifies a spatial bunch length, use `SIGMA_T = SIGMA_Z / scipy.constants.c`.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy.constants import c as c_light

import iddefix
from iddefix.timeDomainVectorFitting import (
    recursive_exponential_convolution,
    sampled_time_derivative,
    time_domain_vector_fit,
)

## User configuration

`TRUNCATION_TIME` is expressed on the physical CST time axis, not on the shifted numerical axis. The resonator bounds are repeated for every resonator; replace them with individual bounds if prior information is available. The wake column must contain one real scalar wake potential.

In [ ]:
# CST file and column convention
DATA_FILE = Path('examples/data/004_SPS_model_transitions_q26.txt')
TIME_COLUMN = 0
WAKE_COLUMN = 2
TIME_SCALE_TO_SECONDS = 1.0e-9
DELIMITER = '\t'

# Physical model
PLANE = 'transverse'  # 'longitudinal' or 'transverse'
SIGMA_T = 1.0e-10     # RMS bunch duration [s]
BUNCH_CENTER_TIME = 0.0  # CST normally centers the bunch at t=0
TRUNCATION_TIME = 8.0e-9  # physical CST time [s], or None for all data

# TD-VF model
NUMBER_OF_REAL_POLES = 1
NUMBER_OF_COMPLEX_PAIRS = 10
FIT_DIRECT_TERM = True
FIT_PROPORTIONAL_TERM = True
MINIMUM_INITIAL_FREQUENCY = 5.0e7
MAXIMUM_INITIAL_FREQUENCY = 3.0e9
TDVF_MAXIMUM_ITERATIONS = 30
TDVF_TOLERANCE = 1.0e-8

# Original IDEFIX resonator model
NUMBER_OF_RESONATORS = 10
RS_BOUNDS = (1.0e-6, 1.0e4)
Q_BOUNDS = (0.51, 500.0)  # analytical wake-potential formula needs Q > 0.5
FREQUENCY_BOUNDS = (5.0e7, 3.0e9)
RESONATOR_MAXIMUM_ITERATIONS = 300
RESONATOR_POPULATION_SIZE = 30
RESONATOR_TOLERANCE = 1.0e-3
RANDOM_SEED = 42

In [ ]:
def create_initial_poles():
    poles = []

    if NUMBER_OF_REAL_POLES:
        decay_rates = np.geomspace(
            2.0 * np.pi * MINIMUM_INITIAL_FREQUENCY,
            2.0 * np.pi * MAXIMUM_INITIAL_FREQUENCY,
            NUMBER_OF_REAL_POLES,
        )
        poles.extend(-decay_rates.astype(complex))

    if NUMBER_OF_COMPLEX_PAIRS:
        frequencies = np.geomspace(
            MINIMUM_INITIAL_FREQUENCY,
            MAXIMUM_INITIAL_FREQUENCY,
            NUMBER_OF_COMPLEX_PAIRS,
        )
        for frequency in frequencies:
            omega = 2.0 * np.pi * frequency
            pole = -0.05 * omega + 1j * omega
            poles.extend([pole, np.conj(pole)])

    return np.asarray(poles, dtype=complex)


def evaluate_tdvf_time_model(times, input_signal, result):
    dt = times[1] - times[0]
    output = result.direct_term * input_signal.astype(complex)

    if result.proportional_term != 0.0:
        output += result.proportional_term * sampled_time_derivative(
            input_signal, dt
        )

    for pole, residue in zip(result.poles, result.residues):
        output += residue * recursive_exponential_convolution(
            input_signal, pole, dt
        )

    return output


def normalized_l2(reference, approximation):
    norm = np.linalg.norm(reference)
    return np.linalg.norm(approximation - reference) / norm if norm else np.nan

## Load and prepare the CST wake potential

The time axis is reconstructed from its mean spacing to avoid failures caused by decimal round-off in text exports. No resampling of the wake values is performed.

In [ ]:
data_candidates = [
    DATA_FILE,
    Path.cwd() / DATA_FILE,
    Path.cwd().parent / DATA_FILE,
]
data_path = next((path.resolve() for path in data_candidates if path.exists()), None)
if data_path is None:
    searched = '\n'.join(str(path.resolve()) for path in data_candidates)
    raise FileNotFoundError(f'CST data file not found. Searched:\n{searched}')
data = np.loadtxt(data_path, comments='#', delimiter=DELIMITER)

physical_times_raw = data[:, TIME_COLUMN] * TIME_SCALE_TO_SECONDS
wake_potential = data[:, WAKE_COLUMN].astype(float)
dt = (physical_times_raw[-1] - physical_times_raw[0]) / (len(physical_times_raw) - 1)
physical_times = physical_times_raw[0] + np.arange(len(physical_times_raw)) * dt
times = physical_times - physical_times[0]

if SIGMA_T <= 0.0:
    raise ValueError('SIGMA_T must be positive.')
if PLANE not in {'longitudinal', 'transverse'}:
    raise ValueError("PLANE must be 'longitudinal' or 'transverse'.")

bunch_profile = np.exp(
    -0.5 * ((physical_times - BUNCH_CENTER_TIME) / SIGMA_T) ** 2
) / (np.sqrt(2.0 * np.pi) * SIGMA_T)

if TRUNCATION_TIME is None:
    fit_mask = np.ones(times.size, dtype=bool)
else:
    fit_mask = physical_times <= TRUNCATION_TIME

if np.count_nonzero(fit_mask) < 3:
    raise ValueError('The selected fit interval contains fewer than three samples.')
if not np.any(fit_mask & (np.abs(physical_times - BUNCH_CENTER_TIME) <= 4.0 * SIGMA_T)):
    raise ValueError('The fit interval does not contain the Gaussian excitation.')

fit_times = times[fit_mask]
fit_physical_times = physical_times[fit_mask]
fit_bunch_profile = bunch_profile[fit_mask]
fit_wake_potential = wake_potential[fit_mask]

print(f'Data file: {data_path}')
print(f'Samples: {times.size}; fit samples: {fit_times.size}')
print(f'dt = {dt:.6e} s')
print(f'sigma_t = {SIGMA_T:.6e} s; sigma_z = {SIGMA_T * c_light:.6e} m')
print(f'Fit ends at physical CST time {fit_physical_times[-1]:.6e} s')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, constrained_layout=True)
axes[0].plot(physical_times * 1e9, bunch_profile / np.max(bunch_profile), color='tab:blue')
axes[0].set_ylabel('Normalized bunch profile')
axes[1].plot(physical_times * 1e9, wake_potential, color='black')
axes[1].set_ylabel('CST wake potential')
axes[1].set_xlabel('Physical CST time [ns]')
for axis in axes:
    axis.axvline(fit_physical_times[-1] * 1e9, color='tab:red', linestyle=':', label='End of fit data')
    axis.grid(True)
    axis.legend()
plt.show()

## Time-Domain Vector Fitting

In [ ]:
initial_poles = create_initial_poles()
tdvf_start = time.perf_counter()
tdvf_result = time_domain_vector_fit(
    times=fit_times,
    input_signal=fit_bunch_profile,
    output_signal=fit_wake_potential,
    initial_poles=initial_poles,
    maximum_iterations=TDVF_MAXIMUM_ITERATIONS,
    tolerance=TDVF_TOLERANCE,
    enforce_stability=True,
    weights=None,
    fit_direct_term=FIT_DIRECT_TERM,
    fit_proportional_term=FIT_PROPORTIONAL_TERM,
)
tdvf_runtime = time.perf_counter() - tdvf_start

tdvf_wake_potential = evaluate_tdvf_time_model(
    times, bunch_profile, tdvf_result
).real

print(f'TD-VF runtime: {tdvf_runtime:.3f} s')
print(f'Relocation iterations: {tdvf_result.iterations}')
print(f'Direct term: {tdvf_result.direct_term:.8e}')
print(f'Proportional term: {tdvf_result.proportional_term:.8e}')

## Original IDEFIX resonator fit

All resonators initially use the same broad bounds. For difficult CST data, narrower resonance-specific bounds or `SmartBoundDetermination` will substantially improve speed and robustness.

In [ ]:
np.random.seed(RANDOM_SEED)
resonator_bounds = []
for _ in range(NUMBER_OF_RESONATORS):
    resonator_bounds.extend([RS_BOUNDS, Q_BOUNDS, FREQUENCY_BOUNDS])

resonator_model = iddefix.EvolutionaryAlgorithm(
    x_data=fit_physical_times,
    y_data=fit_wake_potential,
    N_resonators=NUMBER_OF_RESONATORS,
    parameterBounds=resonator_bounds,
    plane=PLANE,
    fitFunction='wake potential',
    sigma=SIGMA_T,
    objectiveFunction=iddefix.ObjectiveFunctions.sumOfSquaredErrorReal,
)

resonator_start = time.perf_counter()
resonator_model.run_differential_evolution(
    maxiter=RESONATOR_MAXIMUM_ITERATIONS,
    popsize=RESONATOR_POPULATION_SIZE,
    tol=RESONATOR_TOLERANCE,
    mutation=(0.1, 0.5),
    crossover_rate=0.8,
)
resonator_runtime = time.perf_counter() - resonator_start
resonator_wake_potential = resonator_model.get_wake_potential(
    physical_times, sigma=SIGMA_T, use_minimization=False
)
print(f'Resonator runtime: {resonator_runtime:.3f} s')

## Common comparison over fit and extrapolation intervals

In [ ]:
validation_mask = ~fit_mask
models = {
    'TD-VF': tdvf_wake_potential,
    'Resonators': resonator_wake_potential,
}

print('Normalized L2 errors')
print('-' * 68)
for name, model_wake in models.items():
    fit_error = normalized_l2(wake_potential[fit_mask], model_wake[fit_mask])
    full_error = normalized_l2(wake_potential, model_wake)
    validation_error = (
        normalized_l2(wake_potential[validation_mask], model_wake[validation_mask])
        if np.any(validation_mask) else np.nan
    )
    print(f'{name:12s} fit={fit_error:.6e}  validation={validation_error:.6e}  full={full_error:.6e}')

fig, axes = plt.subplots(3, 1, figsize=(12, 11), sharex=True, constrained_layout=True)
time_ns = physical_times * 1e9
end_ns = fit_physical_times[-1] * 1e9

axes[0].plot(time_ns, wake_potential, color='black', linewidth=1.5, label='CST wake potential')
axes[0].plot(time_ns, tdvf_wake_potential, '--', label='TD-VF')
axes[0].plot(time_ns, resonator_wake_potential, ':', linewidth=1.7, label='Resonators')
axes[0].set_title('Complete CST interval: fit and extrapolation')
axes[0].set_ylabel('Wake potential')
axes[0].legend()

axes[1].plot(time_ns[fit_mask], wake_potential[fit_mask], color='black', linewidth=1.5, label='CST')
axes[1].plot(time_ns[fit_mask], tdvf_wake_potential[fit_mask], '--', label='TD-VF')
axes[1].plot(time_ns[fit_mask], resonator_wake_potential[fit_mask], ':', linewidth=1.7, label='Resonators')
axes[1].set_title('Fit interval')
axes[1].set_ylabel('Wake potential')
axes[1].legend()

scale = max(np.max(np.abs(wake_potential)), np.finfo(float).eps)
axes[2].semilogy(time_ns, np.maximum(np.abs(tdvf_wake_potential - wake_potential) / scale, np.finfo(float).eps), label='TD-VF error')
axes[2].semilogy(time_ns, np.maximum(np.abs(resonator_wake_potential - wake_potential) / scale, np.finfo(float).eps), label='Resonator error')
axes[2].set_title('Absolute error normalized by maximum CST wake potential')
axes[2].set_xlabel('Physical CST time [ns]')
axes[2].set_ylabel('Normalized error')
axes[2].legend()

for axis in axes:
    axis.axvline(end_ns, color='tab:red', linestyle=':', label='End of fit data')
    axis.grid(True, which='both')
plt.show()